# Hands-on Lab: Cài đặt và sử dụng Faster-Whisper để nhận dạng giọng nói

**Thời lượng:** 30-45 phút

**Mục tiêu:** cài đặt môi trường, chạy thử Faster-Whisper và hiểu cách model nhận dạng giọng nói thành văn bản. Mục tiêu không phải xây dựng một hệ thống hoàn chỉnh, mà là hiểu quy trình, cấu hình môi trường và cách dùng model ASR đúng cách.

---

## 1. Faster-Whisper là gì?

Faster-Whisper là một triển khai Whisper được tối ưu bằng CTranslate2. Nó giúp tăng tốc quá trình chuyển giọng nói thành văn bản và hỗ trợ:
- nhận dạng giọng nói từ file âm thanh;
- phát hiện ngôn ngữ;
- trả về timestamp cho từng đoạn text;
- chạy trên CPU hoặc GPU nếu môi trường cho phép.

Bài lab này ưu tiên cho máy chạy CPU, vì đây là phương án đơn giản nhất. GPU chỉ là tùy chọn nếu máy có hỗ trợ card đồ họa rời.


## 2. Môi trường khuyến nghị cho lab

Dưới đây là cấu hình phù hợp để làm mục tiêu chính cho lab.

| Thành phần | Phiên bản / phạm vi phù hợp | Bắt buộc | Link chính thức | Ghi chú |
|---|---|---:|---|---|
| Python | 3.10.x | Có | (window)https://www.python.org/downloads/windows/ | Dành cho Windows 10/11 64-bit |
| Faster-Whisper | phiên bản đang được hỗ trợ cùng CTranslate2 | Có | https://github.com/SYSTRAN/faster-whisper | Cài qua pip |
| NVIDIA driver | tương thích với GPU đang dùng | Chỉ khi cần GPU | https://www.nvidia.com/Download/index.aspx | Không bắt buộc nếu chạy CPU |
| CUDA Toolkit 12.x | cần đối chiếu với tài liệu chính thức | Chỉ khi dùng GPU | https://developer.nvidia.com/cuda-12-0-0-download-archive | Không cần nếu chạy CPU |
| cuDNN 9.x | phiên bản tương thích với CUDA đang dùng | Chỉ khi dùng GPU | https://developer.nvidia.com/cudnn-downloads | Không cần nếu chạy CPU |

### Môi trường chính được chọn cho lab
- Hệ điều hành: Windows 10/11 64-bit
- Python: 3.10.x
- Faster-Whisper: phiên bản 1.2.1
- Model được dùng: `small`
- `compute_type`: `int8` cho CPU; `float16` nếu GPU phù hợp, nhưng không bắt buộc duy nhất


## 3. Cài đặt môi trường

### Phương án 1: Máy không có GPU hoặc chỉ cần chạy CPU

Đây là phương án khuyến nghị cho bài lab.

**Bước 1: Cài Python**
- Tải Python 3.10.x từ: https://www.python.org/downloads/windows 

- Hoặc tải trực tiếp Python 3.10.11 qua link: 
    - https://www.python.org/ftp/python/3.10.11/python-3.10.11-amd64.exe (64x)
    - https://www.python.org/ftp/python/3.10.11/python-3.10.11.exe (32x)

![Cài Đặt Python](pics/image.png)

- Chọn `Add Python to PATH`

- Bấm Install

**Bước 2: Chọn môi trường cho notebook**
bấm vào Select Kernel cho Notebook, chọn Python 3.10.x vừa cài đặt.

**Bước 3: Kiểm tra phiên bản Python**


In [ ]:
!python --version

**Bước 4: Cài thư viện cần thiết**

In [ ]:
# Bỏ dấu '#' , rồi chạy nếu lần đầu tiên mở Notebook
!py -m pip install --upgrade pip
%pip install faster-whisper==1.2.1 ipywidgets

**Bước 5: Kiểm tra phiên bản**

In [ ]:
!python -c "import faster_whisper, ctranslate2; print(f'Faster Whisper = {faster_whisper.__version__}'); print(f'CTranslate2 = {ctranslate2.__version__}')"



### Phương án 2 : Máy có GPU NVIDIA và muốn sử dụng GPU

Nếu máy bạn có GPU NVIDIA, hãy kiểm tra phần cứng và driver trước khi dùng GPU cho transcription.

1. Kiểm tra GPU:
```powershell
nvidia-smi
```
2. Kiểm tra driver hiện có.
3. Đối chiếu với tài liệu chính thức của Faster-Whisper và CTranslate2.
4. Chỉ cài CUDA/cuDNN nếu hệ thống chưa có phiên bản phù hợp.
5. Không cần cài lại driver nếu máy đang hoạt động tốt.

**Link chính thức:**
- NVIDIA Driver: https://www.nvidia.com/Download/index.aspx
- CUDA Toolkit: https://developer.nvidia.com/cuda-downloads
- cuDNN: https://developer.nvidia.com/cudnn-downloads

> Yêu cầu CUDA/cuDNN có thể thay đổi tùy phiên bản CTranslate2 đang dùng. Vì vậy, không được cài máy móc; phải đối chiếu với tài liệu chính thức.

**Hoặc có thể dùng pip để tải** (không cần cài CUDA Toolkit/cuDNN riêng):

In [ ]:
# Bỏ dấu '#' rồi chạy nếu muốn cài CUDA (cuBLAS) và cuDNN bằng pip
#!python -m pip install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

In [ ]:
# Kiểm tra phiên bản cuBLAS và cuDNN đã cài bằng pip
from importlib.metadata import version, PackageNotFoundError

for pkg in ['nvidia-cublas-cu12', 'nvidia-cudnn-cu12']:
    try:
        print(f'{pkg} = {version(pkg)}')
    except PackageNotFoundError:
        print(f'{pkg}: chưa cài qua pip (có thể bạn đã cài CUDA Toolkit/cuDNN thủ công)')

# Nếu cài CUDA Toolkit thủ công, kiểm tra thêm bằng lệnh:
print("\n\nCheck gói cài đặt thủ công...")
!nvcc --version


## 4. Bắt đầu với công nghệ Faster-Whisper

### Khởi tạo mô hình cơ bản

Ở bước này, ta tải model Whisper vào bộ nhớ để dùng cho các bước nhận dạng giọng nói tiếp theo.

**Lưu ý:** ở lần chạy đầu tiên, Faster-Whisper sẽ tự tải model về máy, nên bạn cần kết nối mạng và phải chờ một lúc. Từ lần chạy sau, model được lấy lại từ bộ nhớ đệm nên sẽ nhanh hơn.

**Các tham số của `WhisperModel`**

| Tham số | Ý nghĩa | Giá trị dùng trong lab |
|---|---|---|
| `model_size_or_path` (tham số đầu tiên) | Tên hoặc kích thước model cần tải. Model càng lớn thì càng chính xác nhưng càng chậm và tốn RAM. Các cỡ phổ biến: `tiny`, `base`, `small`, `medium`, `large-v3` | `"small"` (cân bằng giữa tốc độ và độ chính xác) |
| `device` | Thiết bị chạy model: `"cpu"`, `"cuda"` (GPU NVIDIA) hoặc `"auto"` | `"cpu"`, hoặc `"cuda"` nếu có GPU |
| `compute_type` | Kiểu số dùng khi tính toán. Kiểu nhẹ hơn thì chạy nhanh hơn và tốn ít bộ nhớ hơn, nhưng độ chính xác có thể giảm nhẹ. Ví dụ: `"int8"`, `"float16"`, `"float32"` | `"int8"` cho CPU, `"int8","float16"` cho GPU |


Hãy chọn cấu hình phù hợp với máy của bạn. Nếu dùng GPU mà gặp lỗi thiếu file DLL, hãy quay lại mục 3 để kiểm tra phiên bản cuBLAS và cuDNN.


In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(model_size_or_path="small", 
                     device="cpu", #Chỉnh thành cuda nếu muốn chạy GPU
                     compute_type="int8") #Nếu xài GPU và muốn chuẩn hơn thì xài float16 tốn tài nguyên hơn
                    

print("Đã khởi tạo model xong")


### Tải dữ liệu test

Bài lab dùng một số file âm thanh mẫu. Cell dưới sẽ tải chúng vào thư mục `audio_test` (bỏ qua file đã có).


In [ ]:
import os, urllib.request

os.makedirs("audio_test", exist_ok=True)

FW = "https://github.com/SYSTRAN/faster-whisper/raw/master/tests/data"
files = {
    "jfk.flac":         f"{FW}/jfk.flac",
    "hotwords.mp3": f"{FW}/hotwords.mp3",
    "physicsworks.wav": f"{FW}/physicsworks.wav",
    "ted_60s.wav":      "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/ted_60.wav",
}

for name, url in files.items():
    path = os.path.join("audio_test", name)
    if os.path.exists(path):
        print(f"Đã có: {name}")
        continue
    req = urllib.request.Request(url, headers={"User-Agent": "faster-whisper-lab"})
    with urllib.request.urlopen(req) as r, open(path, "wb") as f:
        f.write(r.read())
    print(f"Đã tải: {name}")


## 5. Nhận dạng giọng nói đầu tiên

Sau khi đã có `model`, ta dùng hàm `transcribe()` để chuyển một file âm thanh thành văn bản. Ở bước này ta thử với `jfk.flac`, một đoạn ghi âm ngắn khoảng 11 giây, để kết quả có nhanh.

Hàm `transcribe()` trả về hai thứ:

| Giá trị trả về | Ý nghĩa |
|---|---|
| `segments` | Các đoạn văn bản đã nhận dạng. Mỗi đoạn có thời điểm bắt đầu (`start`), kết thúc (`end`) và nội dung (`text`) |
| `info` | Thông tin chung về file: ngôn ngữ phát hiện được (`language`), độ tin cậy (`language_probability`) và độ dài audio (`duration`) |

**Lưu ý:** `segments` là một *generator*, nghĩa là việc nhận dạng chỉ thực sự chạy khi ta duyệt qua nó bằng vòng `for`. Vì vậy cell bên dưới có thể mất vài giây ở vòng lặp, chứ không phải ở dòng gọi `transcribe()`.


In [ ]:
segments, info = model.transcribe("audio_test/jfk.flac")

print(f"Ngôn ngữ: {info.language} (xác suất {info.language_probability:.2f})")
print(f"Độ dài audio: {info.duration:.1f}s\n")

for segment in segments:
    print(f"[{segment.start:6.2f}s -> {segment.end:6.2f}s] {segment.text}")


## 6. Bộ lọc khoảng lặng (VAD)

Trong một file ghi âm thật, người nói thường ngừng nghỉ. **VAD** (Voice Activity Detection) là bước phát hiện đoạn nào có tiếng nói, đoạn nào chỉ là khoảng lặng hoặc tiếng ồn, để model không phải xử lý những đoạn đó. Faster-Whisper có sẵn VAD, bật bằng tham số `vad_filter=True`.

| Tham số | Ý nghĩa |
|---|---|
| `vad_filter` | `True`: loại bỏ khoảng lặng trước khi nhận dạng. `False` (mặc định): đưa toàn bộ audio vào model |
| `vad_parameters` | Tinh chỉnh VAD, ví dụ `dict(min_silence_duration_ms=500)`: chỉ coi là khoảng lặng nếu im lặng từ 500 ms trở lên |

Ta sẽ chạy cùng một file `ted_60s.wav` (60 giây, có nhiều chỗ ngừng nghỉ) với hai chế độ, rồi so sánh kết quả.


In [ ]:
import time

def Vad_Filter_Controller(vad):
    t0 = time.time()
    segments, info = model.transcribe("audio_test/ted_60s.wav", vad_filter=vad)
    segments = list(segments)   # bọc list để quá trình nhận dạng chạy thật sự ở đây
    return segments, info, time.time() - t0

ket_qua = {}
for vad in (False, True):
    segments, info, RunTime = Vad_Filter_Controller(vad)
    ket_qua[vad] = segments
    print(f"vad_filter={vad}: {len(segments)} segment | {RunTime:.1f}s xử lý | "
          f"audio sau VAD: {info.duration_after_vad:.1f}s / {info.duration:.1f}s\n\n Hoàn thành!!")


In [ ]:
for vad, segs in ket_qua.items():
    print(f"\n===== vad_filter={vad}: {len(segs)} segment =====")
    truoc = 0.0
    for i, s in enumerate(segs, 1):
        cach = s.start - truoc   # khoảng cách tới segment trước
        print(f"{i:3d}. [{s.start:7.2f} -> {s.end:7.2f}] dài {s.end - s.start:5.1f}s | cách trước {cach:5.2f}s | {s.text.strip()}")
        truoc = s.end


### Nhận xét

Chạy `ted_60s.wav` với hai chế độ cho thấy:

- VAD giúp **loại bỏ các đoạn không phải lời nói**:
- Các segment ngắn được **gộp lại**, và các chỗ ngừng ngắn nằm *bên trong* segment thay vì tạo khoảng trống giữa hai segment.
- Lời nói gần như không đổi, chỉ khác về dấu câu.

**Lưu ý:** vì khoảng lặng bị gộp vào segment, thời điểm bắt đầu/kết thúc của một số segment có thể lệch so với lúc người nói thực sự cất lời hoặc dừng.


## 7. Timestamp từng từ

Ở các mục trước, mỗi `segment` chỉ có thời điểm bắt đầu và kết thúc của cả câu. Nếu bật `word_timestamps=True`, mỗi segment còn có danh sách `words`, cho biết **từng từ** bắt đầu và kết thúc lúc nào. Thông tin này hữu ích khi cần hiển thị phụ đề khớp từng từ hoặc kiểm tra chính xác thời điểm người nói cất lời.

| Thuộc tính của mỗi từ | Ý nghĩa |
|---|---|
| `word` | Nội dung từ |
| `start`, `end` | Thời điểm bắt đầu và kết thúc (giây) |
| `probability` | Độ tin cậy của model với từ đó (0 đến 1) |

**Lưu ý:** bật `word_timestamps=True` làm quá trình nhận dạng chậm hơn một chút.


In [ ]:
segments, info = model.transcribe("audio_test/ted_60s.wav", word_timestamps=True)

for segment in segments:
    for word in segment.words:
        print(f"[{word.start:5.2f}s -> {word.end:5.2f}s] {word.word.strip():<12} (tin cậy {word.probability:.2f})")


## 8. Một số tham số hay dùng

Hàm `transcribe()` có nhiều tham số để điều chỉnh kết quả. Ở đây ta chỉ làm quen với ba tham số phổ biến:

| Tham số | Ý nghĩa | Đánh đổi |
|---|---|---|
| `beam_size` | Số hướng nhận dạng model thử song song (mặc định 5). `1` là chọn ngay phương án đầu tiên | Giá trị lớn thường chính xác hơn nhưng chậm hơn |
| `initial_prompt` | Một đoạn văn mẫu đưa cho model trước khi nhận dạng, để gợi ý văn phong, từ vựng, cách viết, ngữ cảnh | Chỉ gợi ý, không đảm bảo model làm theo |
| `hotwords` | Chuỗi các từ khóa mà bạn muốn model ưu tiên nhận dạng (tên riêng, thuật ngữ) | Chỉ gợi ý, không đảm bảo |

Phần dưới thử từng tham số với file mẫu.


In [ ]:
import time

def chay(path, **kwargs):
    t0 = time.time()
    segments, info = model.transcribe(path, **kwargs)
    segments = list(segments)   # bọc list để nhận dạng chạy thật sự ở đây
    return segments, time.time() - t0

van_ban = {}
for b in (1, 5):
    segments, giay = chay("audio_test/ted_60s.wav", beam_size=b)
    van_ban[b] = " ".join(s.text.strip() for s in segments)
    print(f"beam_size={b}: {len(segments)} segment | {giay:.1f}s")
print(van_ban[1])
print("\n",van_ban[5])
print("\nVăn bản giống nhau:", van_ban[1] == van_ban[5])


### Thử `hotwords`

Chạy cell dưới để xem model nhận dạng `hotwords.mp3` (file rất ngắn, khoảng 4 giây) khi **chưa** có gợi ý. Nghe file, tìm một từ model nhận dạng sai hoặc viết khác ý bạn, rồi điền vào cell kế tiếp.


In [ ]:
segments, _ = model.transcribe("audio_test/hotwords.mp3")
for s in segments:
    print(s.text.strip())


In [ ]:
tu_khoa = "ComfyUI"   # điền từ bạn muốn model ưu tiên

segments, _ = model.transcribe("audio_test/hotwords.mp3", hotwords=tu_khoa)
for s in segments:
    print(s.text.strip())


**Câu hỏi:** Kết quả có khác sau khi thêm `hotwords` không? Thử thay bằng `initial_prompt="..."` (một câu chứa từ đó VD: "Today we talk about ComfyUI") và so sánh?

In [ ]:
#code here

<details>
<summary>Xem đáp án</summary>

```python
segments, _ = model.transcribe("audio_test/hotwords.mp3",
                               initial_prompt="Today we talk about ComfyUI.")
print(" ".join(s.text.strip() for s in segments))
```

</details>


**Trả lời câu hỏi:**



<details>
<summary>Xem đáp án</summary>

```
Có khác. Khi không gợi ý, model nghe ra ConfUi ở cả hai chỗ. Sau khi thêm hotwords (hoặc initial_prompt), cả hai chỗ đều thành ComfyUI.
hotwords và initial_prompt đều sửa được từ khóa, nhưng khác nhau ở chỗ khác. Với hotwords, ngoài từ ComfyUI thì phần còn lại của câu giống hệt lần không gợi ý. Với initial_prompt, câu có thêm and (and again), tức là phần còn lại của câu cũng bị ảnh hưởng.
initial_prompt làm model viết theo cách viết trong câu gợi ý. Viết comfyUI thì kết quả ra comfyUI, viết ComfyUI thì ra ComfyUI.
```

</details>


## 9. Xuất phụ đề SRT

**SRT** là định dạng phụ đề phổ biến, hầu hết trình phát video đều đọc được. Mỗi đoạn phụ đề gồm 4 phần:

1

00:00:00,000 --> 00:00:04,880

So in college, I was a government major,




| Phần | Ý nghĩa |
|---|---|
| Dòng 1 | Số thứ tự segment, bắt đầu từ 1 |
| Dòng 2 | Thời gian bắt đầu `-->` thời gian kết thúc, theo dạng `giờ:phút:giây,mili giây` |
| Dòng 3 | Nội dung |
| Dòng trống | Ngăn cách với đoạn tiếp theo |

Mỗi `segment` ta đã có ở các mục trước cung cấp đủ `start`, `end` và `text`, nên chỉ cần đổi giây sang đúng định dạng thời gian rồi ghi ra file.


In [ ]:
from faster_whisper import utils
segments, _ = model.transcribe("audio_test/ted_60s.wav")

with open("ted_60s.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        f.write(f"{i}\n{(seg.start)} --> {utils.format_timestamp(seg.end)}\n{seg.text.strip()}\n\n")

print(open("ted_60s.srt", encoding="utf-8").read())


## 10. Bài tập

Thử sức với các yêu cầu sau. Bạn có thể dùng lại code ở các mục trước.

1. **Đổi model:** khởi tạo model `base` thay vì `small`, nhận dạng `jfk.flac` và so sánh kết quả cùng thời gian chạy.
2. **Thử VAD với tham số:** chạy `ted_60s.wav` với `vad_parameters=dict(min_silence_duration_ms=500)`. Số segment và `audio sau VAD` thay đổi thế nào so với lúc dùng mặc định?
3. **File của bạn:** ghi âm một câu ngắn (bằng điện thoại), nhận dạng và xuất thành file SRT. Kết quả có chỗ nào sai không? Vì sao?

In [ ]:
#Code here

## Đáp án bài tập

<details>
<summary>Bài 1: đổi sang model `base`</summary>

```python
from faster_whisper import WhisperModel
import time

for ten in ("small", "base"):
    m = WhisperModel(ten, device="cpu", compute_type="int8")
    t0 = time.time()
    segments, _ = m.transcribe("audio_test/jfk.flac")
    van_ban = " ".join(s.text.strip() for s in segments)
    print(f"{ten}: {time.time() - t0:.1f}s | {van_ban}")
```

Khi mình chạy trên `jfk.flac`, cả hai model cho **cùng một câu**, nhưng `base` nhanh hơn (khoảng 2,5 giây so với 8,1 giây của `small` tốc độ sẽ khác nhau tùy vào phần cứng). Đây là một file ngắn và rõ nên hai model không phân biệt được. Với giọng khó nghe hoặc nền ồn, model nhỏ thường sai nhiều hơn.

Lần chạy `small` ở đây có thể gồm cả thời gian "khởi động", nên đừng kết luận chính xác con số.
</details>

<details>
<summary>Bài 2: VAD với `min_silence_duration_ms=500`</summary>

```python
for ten, kw in (("mặc định", {}),
                ("min_silence=500", {"vad_parameters": dict(min_silence_duration_ms=500)})):
    segments, info = model.transcribe("audio_test/ted_60s.wav", vad_filter=True, **kw)
    segments = list(segments)
    print(f"{ten}: {len(segments)} segment | audio sau VAD {info.duration_after_vad:.1f}s / {info.duration:.1f}s")
```

Khi tôi chạy:

| | Số segment | Audio sau VAD |
|---|---|---|
| Mặc định | 21 | 55,8s / 60,0s |
| `min_silence_duration_ms=500` | 22 | 53,0s / 60,0s |

Hạ ngưỡng xuống 500 ms thì VAD **cắt nhiều khoảng lặng hơn** (còn 53,0 giây thay vì 55,8 giây), vì các chỗ ngừng ngắn hơn cũng bị coi là khoảng lặng. Số segment chỉ thay đổi ít.

Số segment ở lần chạy "mặc định" của tôi (21) khác với con số trong notebook của bạn (22), vì kết quả có thể khác nhau giữa các lần chạy. Con số `audio sau VAD` thì ổn định hơn.
</details>

<details>
<summary>Bài 3: file ghi âm của bạn</summary>

Không có đáp án cố định, vì kết quả phụ thuộc vào bản ghi âm của bạn. Một số lỗi thường gặp khi kiểm tra:
- **Tên riêng, thuật ngữ** bị viết sai (như `ConfUi` thay cho `ComfyUI` ở mục 8). Thử `hotwords` hoặc `initial_prompt`.
- **Nói nhanh, nhỏ tiếng hoặc có tiếng ồn nền:** model có thể bỏ sót hoặc nghe nhầm từ. Từ có `tin cậy` thấp (xem mục 7) thường là chỗ cần kiểm tra trước.
- **Chỗ ngừng dài:** nếu không bật VAD, model có thể sinh chữ ở đoạn im lặng, hoặc timestamp bị lệch (xem mục 6).
- **Sai ngôn ngữ phát hiện:** kiểm tra `info.language`. Nếu sai, truyền `language="vi"` (hoặc ngôn ngữ đúng).

Hãy đối chiếu với audio gốc trước khi tin vào văn bản.
</details>
